# Python to TypeScript Code Generator

Use a frontier LLM via **OpenRouter only** to generate idiomatic, typed TypeScript code from Python that produces equivalent behavior.

**Setup:** Set `OPENROUTER_API_KEY` in your `.env` file. Get a key at [openrouter.ai](https://openrouter.ai).

**Run TypeScript:** The "Run TypeScript" button requires Node.js and `ts-node` (or `tsx`) installed, e.g. `npm i -g ts-node` or use `npx ts-node main.ts`.

## Imports

Dependencies: `dotenv` for API key, `openai` for the OpenRouter client, `gradio` for the UI.

In [21]:
# imports
import os
import io
import sys
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

## API key

Load `OPENROUTER_API_KEY` from `.env`. Get a key at [openrouter.ai](https://openrouter.ai).

In [22]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set - please add OPENROUTER_API_KEY to your .env file")

OpenRouter API Key exists and begins sk-or-v1


## OpenRouter client and models

Single client for OpenRouter; model dropdown uses OpenRouter model IDs (e.g. GPT-4o, Claude, Gemini).

In [23]:
# Connect to OpenRouter - single client for all models
client = OpenAI(
    api_key=openrouter_api_key,
    base_url=OPENROUTER_BASE_URL,
)

# Model dropdown: OpenRouter model IDs
MODELS = [
    "openai/gpt-4o",
    "anthropic/claude-3-5-sonnet",
    "google/gemini-2.0-flash-001",
    "openai/gpt-4o-mini",
    "anthropic/claude-3-haiku",
]

## Prompts for TypeScript

System prompt asks for idiomatic TypeScript with types; user prompt wraps the Python code. `messages_for(python)` builds the chat message list.

In [24]:
system_prompt = """
Your task is to convert Python code into idiomatic TypeScript.
Use proper types, interfaces, and type annotations. The TypeScript code must produce the same runtime behavior and output as the Python code.
Respond only with TypeScript code. Do not wrap the code in markdown code fences or provide explanations.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to equivalent TypeScript. Use proper types and idiomatic TypeScript. The code will be run with ts-node.
Respond only with TypeScript code.

Python code to port:

```python
{python}
```
"""

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

## Write output

Writes the generated TypeScript to a file (default `main.ts`, UTF-8).

In [25]:
def write_output(ts_code, path="main.ts"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(ts_code)

## Port (convert)

Calls the LLM with `messages_for(python)`, strips markdown code fences from the reply, writes to file, and returns the TypeScript for display.

In [26]:
def port(model, python):
    kwargs = {"model": model, "messages": messages_for(python)}
    if 'gpt' in model:
        kwargs["reasoning_effort"] = "high"
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content or ""
    # Strip markdown code fences
    reply = reply.strip()
    for prefix in ('```typescript', '```ts', '```'):
        if reply.startswith(prefix):
            reply = reply[len(prefix):].lstrip('\n')
            break
    if reply.endswith('```'):
        reply = reply[:-3].rstrip()
    write_output(reply)
    return reply

## Run Python

Executes the Python code in a sandbox, captures stdout, and returns it (or an error message).

In [27]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output

## Run TypeScript

Writes the TypeScript to a temp file, runs it with `npx tsx` (or `ts-node`), and returns stdout (or stderr on failure). Requires Node.js and tsx/ts-node.

In [28]:
def run_typescript(ts_code):
    if not ts_code or not ts_code.strip():
        return "No TypeScript code to run. Convert Python first."
    cwd = os.getcwd()
    run_file = os.path.join(cwd, "_gradio_ts_run.ts")
    try:
        with open(run_file, "w", encoding="utf-8") as f:
            f.write(ts_code)
        # Prefer tsx (captures stdout reliably); fall back to ts-node
        for cmd in [["npx", "--yes", "tsx", run_file], ["npx", "--yes", "ts-node", run_file]]:
            try:
                result = subprocess.run(
                    cmd,
                    cwd=cwd,
                    capture_output=True,
                    text=True,
                    encoding="utf-8",
                    timeout=30,
                    env={**os.environ, "FORCE_COLOR": "0"},
                )
                break
            except FileNotFoundError:
                continue
        else:
            return "Neither tsx nor ts-node found. Install with: npx tsx or npm i -g ts-node"
        if result.returncode != 0:
            return result.stderr or result.stdout or "Non-zero exit"
        out = (result.stdout or "").strip()
        err = (result.stderr or "").strip()
        if out:
            return out
        if err:
            return err
        return "(Program ran successfully with no output.)"
    except subprocess.TimeoutExpired:
        return "Execution timed out."
    except Exception as e:
        return f"Error: {e}"
    finally:
        if os.path.exists(run_file):
            try:
                os.unlink(run_file)
            except OSError:
                pass

## Default example

Short Python snippet shown in the first code box so you can click Convert immediately.

In [29]:
DEFAULT_PYTHON = '''print("Hello from Python")
x = [1, 2, 3]
print(sum(x))'''

## Gradio UI

Two code areas (Python input, TypeScript output), model dropdown, Convert button, and Run Python / Run TypeScript with result text areas. Launch in browser.

In [31]:
with gr.Blocks() as ui:
    gr.Markdown("## Python to TypeScript - By Sheriff Ibrahm")
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=16, value=DEFAULT_PYTHON)
        ts = gr.Textbox(label="TypeScript code:", lines=16)
    with gr.Row():
        model = gr.Dropdown(MODELS, label="Model", value=MODELS[0])
        convert_btn = gr.Button("Convert")
    convert_btn.click(port, inputs=[model, python], outputs=[ts])

    with gr.Row():
        py_result = gr.Textbox(label="Python output", lines=4)
        ts_result = gr.Textbox(label="TypeScript output", lines=4)
    with gr.Row():
        run_py_btn = gr.Button("Run Python")
        run_ts_btn = gr.Button("Run TypeScript")
    run_py_btn.click(run_python, inputs=[python], outputs=[py_result])
    run_ts_btn.click(run_typescript, inputs=[ts], outputs=[ts_result])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
